# 02 - Analyse du risque de taux et de spread

À lancer **souvent** : tout est relu depuis les caches créés par `01_collecte.ipynb`, en quelques secondes.

1. Obligations encore vivantes
2. Prix, durations, convexité, key rate durations (une ligne par tranche)
3. Synthèse par secteur
4. Stress tests : choc parallèle, scénarios de déformation de courbe, choc de spread
5. Exports : `data/resultats.csv` (pour l'application), `data/rapport_resultats.xlsx`, `data/rapport.html`

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

from courbe import get_courbe_bam
from pipeline import (charger_tranches, obligations_vivantes, analyser_marche, resume_secteurs,
                      impact_scenario, tous_les_scenarios, FICHIER_RESULTATS)
from calculs import SCENARIOS
from rapport import exporter_excel, exporter_html

courbe = get_courbe_bam()          # relit data/courbe_bam.csv
tranches = charger_tranches()      # relit data/tranches.csv
print("Courbe du", pd.to_datetime(courbe["date_valeur"]).iloc[0].date(), "|", len(tranches), "tranches")

## 1. Obligations encore vivantes

In [ ]:
vivantes = obligations_vivantes(tranches)
print(len(vivantes), "tranches vivantes, complètes et sans doublon")
print(vivantes["nature_taux"].value_counts().to_string())
print("Échéances estimées (visa + maturité) :", int(vivantes["echeance_estimee"].sum()))

## 2. Analyse de chaque tranche

Flux datés en fractions d'année, chacun actualisé à **courbe BAM(t) + prime**. Colonnes principales :
`prix_plein`, `coupon_couru`, `prix_pied`, `macaulay`, `modifiee` (duration de taux effective), `convexite`,
`duration_spread`, `dv01` (MAD par titre de 100 000), `seuil_pb`, chocs parallèles et `KRD …` (key rate durations).

In [ ]:
resultats = analyser_marche(vivantes, courbe)
resultats.to_csv(FICHIER_RESULTATS, index=False)       # lu par l'application Streamlit
resultats.head(10)

## 3. Synthèse par secteur

In [ ]:
resume_secteurs(resultats)

In [ ]:
pd.crosstab(resultats["secteur"], resultats["nature_taux"], margins=True, margins_name="Total")

### Tranches à taux fixe : les plus résistantes et les plus exposées

In [ ]:
fixes = resultats[resultats["nature_taux"] == "fixe"]
colonnes = ["emetteur", "tranche", "secteur", "maturite_res", "ytm_%", "modifiee", "convexite", "seuil_pb", "choc_+100pb_%"]
print("Plus résistantes :"); display(fixes.nsmallest(5, "modifiee")[colonnes])
print("Plus exposées :");    display(fixes.nlargest(5, "modifiee")[colonnes])

## 4. Scénarios de déformation de la courbe

Chocs en points de base aux points de la courbe, interpolés entre les points (inspirés des scénarios IRRBB de Bâle).

In [ ]:
pd.DataFrame(SCENARIOS, index=["3 mois", "1 an", "2 ans", "5 ans", "10 ans", "15 ans", "30 ans"]).T

In [ ]:
scenarios = tous_les_scenarios(resultats, courbe)
scenarios.groupby("secteur")[list(SCENARIOS)].mean().round(2)

In [ ]:
scenarios.groupby("nature_taux")[list(SCENARIOS)].mean().round(2)

### Choc de spread

La prime de risque ne se révise jamais : une tranche révisable est protégée contre les taux de BAM, **pas** contre une hausse de la prime de son émetteur. Exemple : +100 pb sur l'immobilier, +50 pb sur les sociétés de financement.

In [ ]:
spread = impact_scenario(resultats, courbe, chocs_pb=None,
                         spreads_pb={"Immobilier": 100, "Société de financement": 50})
spread.pivot_table(index="secteur", columns="nature_taux", values="variation_%", aggfunc="mean").round(2)

## 5. Exports

In [ ]:
print(exporter_excel(resultats, courbe))
print(exporter_html(resultats, courbe))

## Conclusions

*À compléter avec tes chiffres :*

- **Structure du marché** : … % des tranches vivantes sont à taux révisable ; les banques représentent … tranches sur ….
- **Risque de taux** : duration de taux moyenne de … ; le risque se concentre sur … (secteur, maturité).
- **Risque de crédit** : prime moyenne de … pb (banques) à … pb (immobilier).
- **Scénarios** : le scénario le plus pénalisant pour le marché est … ; les tranches fixes de moyen terme …
- **Spread** : les révisables ont une duration de taux d'environ … mais une duration de spread de …